# Data Source Scaling Comparison

Scaling curves for four training data sources — QF-only, LI-only, their
deduplicated Union, and Unfiltered — all trained with the same schedule
and model capacities. States subsampled with `drop_prob=0.9`; variance
across seeds comes from different random state drops.

Schedule: 2K -> 200K games, 5 seeds per scale point.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Load all four result sets
sources = {}
for name, path in [
    ('Union', 'combined_union_scaling_results.json'),
    ('QF-only', 'qf_only_scaling_results.json'),
    ('LI-only', 'li_only_scaling_results.json'),
    ('Unfiltered', 'unfiltered_scaling_results.json'),
]:
    with open(path) as f:
        d = json.load(f)
    grp = defaultdict(list)
    for run in d['runs']:
        key = (run['max_games'], run['num_leaves'], run['num_trees'])
        grp[key].append(run)
    sources[name] = grp

# Use union's keys as reference
ref_keys = sorted(sources['Union'].keys())
ref_game_counts = [k[0] for k in ref_keys]
drop_prob = d.get('drop_prob', 'N/A')
n_seeds = max(len(sources['Union'][k]) for k in ref_keys)

print(f'Loaded {len(sources)} sources, {len(ref_keys)} scale points, '
      f'{n_seeds} seeds, drop_prob={drop_prob}')

In [ ]:
def fmt(vals):
    return f'{np.mean(vals):.4f} +/- {np.std(vals):.4f}'

print(f'{"Source":>12} {"Games":>8}  {"Log Loss":>18}  {"Accuracy":>18}  '
      f'{"AUC":>18}  {"Egg":>18}  {"SymDev":>18}')
print('-' * 125)

for name in ['Union', 'QF-only', 'LI-only', 'Unfiltered']:
    grp = sources[name]
    for key in ref_keys:
        mg = key[0]
        runs = grp.get(key, [])
        if not runs:
            continue
        loss = [r['metrics']['standard']['log_loss'] for r in runs]
        acc = [r['metrics']['standard']['accuracy'] for r in runs]
        auc = [r['metrics']['standard']['auc_roc'] for r in runs]
        egg = [r['metrics']['kq']['egg_inversion_rate'] for r in runs]
        sym = [r['metrics']['kq']['symmetry_deviation'] for r in runs]
        print(f'{name:>12} {mg:>8}  {fmt(loss)}  {fmt(acc)}  '
              f'{fmt(auc)}  {fmt(egg)}  {fmt(sym)}')
    print()

In [ ]:
colors = {'Union': '#2196F3', 'QF-only': '#4CAF50', 'LI-only': '#FF9800', 'Unfiltered': '#9C27B0'}
markers = {'Union': 'o', 'QF-only': 's', 'LI-only': '^', 'Unfiltered': 'D'}

compare_metrics = [
    ('standard', 'log_loss', 'Log Loss'),
    ('standard', 'accuracy', 'Accuracy'),
    ('standard', 'auc_roc', 'AUC-ROC'),
    ('kq', 'egg_inversion_rate', 'Egg Inversion Rate'),
    ('kq', 'symmetry_deviation', 'Symmetry Deviation'),
    ('calibration', 'ece', 'ECE'),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for ax, (suite, metric_key, label) in zip(axes, compare_metrics):
    for name in ['Union', 'QF-only', 'LI-only', 'Unfiltered']:
        grp = sources[name]
        means, stds = [], []
        for key in ref_keys:
            runs = grp.get(key, [])
            vals = [r['metrics'][suite][metric_key] for r in runs]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        means, stds = np.array(means), np.array(stds)

        ax.errorbar(ref_game_counts, means, yerr=stds,
                    marker=markers[name], capsize=4, linewidth=2,
                    color=colors[name], label=name)
        ax.fill_between(ref_game_counts, means - stds, means + stds,
                        alpha=0.1, color=colors[name])

    ax.set_xlabel('Games')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_xscale('log', base=2)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    ax.set_xticks(ref_game_counts)
    ax.set_xticklabels([f'{mg//1000}K' for mg in ref_game_counts], fontsize=8)

fig.suptitle('Union vs QF-only vs LI-only vs Unfiltered Scaling\n'
             f'drop_prob={drop_prob}, {n_seeds} seeds, tournament holdout',
             fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('source_comparison_plot.png', dpi=200, bbox_inches='tight')
print('Saved source_comparison_plot.png')
plt.show()